In [1]:
import pandas as pd
import numpy as np 
import os
import json

In [52]:
EXP_NUMBER = 3
NUM_DATA_POINTS = 50000

**load the data**

In [53]:

#read the json file
data_file_path = "../data/processed/multinli_1.0_train_cleaned.csv"
data = pd.read_csv(data_file_path, sep='µ')

#get a random sample of the data
data = data.sample(n = NUM_DATA_POINTS, random_state = 42)
data.head()

C:\Users\hamma\AppData\Local\Temp\ipykernel_25664\1779557940.py:3: ParserWarning: Falling back to the 'python' engine because the separator encoded in utf-8 is > 1 char long, and the 'c' engine does not support such separators; you can avoid this warning by specifying engine='python'.
  data = pd.read_csv(data_file_path, sep='µ')


,annotator_labels,genre,gold_label,sentence1,sentence2
299696,neutral,telephone,neutral,right and try to determine if what you read wa...,What you read was definitely biased.
68324,neutral,fiction,neutral,"An impatient voice cried ""Come in"" in answer t...",The voice said to come in and they continued t...
41847,neutral,slate,neutral,All ideological differences between Democrats ...,The difference in ideology between Republicans...
321185,contradiction,slate,contradiction,In instances where Clinton has asserted the pr...,The Republicans have no trouble with Clinton.
344052,entailment,telephone,entailment,i guess um we're assuming that that Puerto Ric...,I assume that Puerto Rico would be overall poo...


### Apply TF-IDF

In [54]:
import spacy
import re
import unicodedata
from tqdm import tqdm

def clean_text(text):
    # Remove accented characters (optional)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    # Remove anything that is NOT a letter or space
    text = re.sub(r"[^a-zA-Z\s]", ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Lowercase the text
    text = text.lower()
    
    return text

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]  # disable unused components
)


def preprocess_texts(texts, batch_size=10000):
    cleaned = []

    # First, clean numbers/special chars
    texts = [clean_text(t) for t in texts]

    for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts)):
        tokens = [
            token.lemma_.lower()
            for token in doc
        ]
        cleaned.append(" ".join(tokens))

    return cleaned

In [55]:
preprocess_texts(['not no  does, never, neither, nobody, nothing ,nowhere, none, nor, cannot, t, without, hardly, scarcely, barely, seldom, rarely trying to test the code. All every each always completely entirely totally definitely certainly surely'])

100%|██████████| 1/1 [00:00<00:00, 576.06it/s]


['not no do never neither nobody nothing nowhere none nor can not t without hardly scarcely barely seldom rarely try to test the code all every each always completely entirely totally definitely certainly surely']

In [56]:
#clean text sentence1 and sentence2
data['sentence1_cleaned'] = preprocess_texts(data['sentence1'].astype(str).tolist())
data['sentence2_cleaned'] = preprocess_texts(data['sentence2'].astype(str).tolist())

100%|██████████| 50000/50000 [00:35<00:00, 1402.94it/s]


In [57]:
#apply tf-idf, remove stop words, remove numbering
from sklearn.feature_extraction.text import TfidfVectorizer

all_sentences = data['sentence1_cleaned'].tolist() + data['sentence2_cleaned'].tolist()

# Fit on all text to get the same vocabulary
tfidf = TfidfVectorizer(min_df=0.001, ngram_range=(1, 2))
tfidf.fit(all_sentences)

# Transform each column separately using the same vectorizer
tfidf_s1_matrix = tfidf.transform(data['sentence1_cleaned']).toarray()
tfidf_s2_matrix = tfidf.transform(data['sentence2_cleaned']).toarray()

In [58]:
#use the tfidf vectors as features (concatenate sentence1 and sentence2)
feature_names_s1 = [f"s1_{w}" for w in tfidf.get_feature_names_out()]
feature_names_s2 = [f"s2_{w}" for w in tfidf.get_feature_names_out()]
all_feature_names = feature_names_s1 + feature_names_s2

features = pd.DataFrame(
    np.hstack([tfidf_s1_matrix, tfidf_s2_matrix]),
    columns=all_feature_names
)

#add the features to the cleaned_data
data_tf_idf = pd.concat([data.reset_index(drop=True), features.reset_index(drop=True)], axis=1)

#drop sentence1 and sentence2
data_tf_idf = data_tf_idf.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])
data_tf_idf.head()

,gold_label,sentence1_cleaned,sentence2_cleaned,s1_ability,s1_ability to,s1_able,s1_able to,s1_about,s1_about it,s1_about that,...,s2_you should,s2_you think,s2_you to,s2_you ve,s2_you want,s2_you will,s2_you you,s2_young,s2_your,s2_yourself
0,neutral,right and try to determine if what you read be...,what you read be definitely biased,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,neutral,an impatient voice cry come in in answer to th...,the voice say to come in and they continue thr...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,neutral,all ideological difference between democrats a...,the difference in ideology between republicans...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,contradiction,in instance where clinton have assert the priv...,the republicans have no trouble with clinton,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,entailment,i guess um we re assume that that puerto rico ...,i assume that puerto rico would be overall poo...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
len(all_feature_names) / 2

2576.0

In [60]:
#select from this list names that start with 't'
t_features = [name for name in all_feature_names if name.startswith('s1_do')]
print(f"Total features starting with 't': {len(t_features)}")
print(t_features)  # show first 20


Total features starting with 't': 31
['s1_do', 's1_do it', 's1_do not', 's1_do so', 's1_do something', 's1_do that', 's1_do the', 's1_do they', 's1_do this', 's1_do with', 's1_do you', 's1_doctor', 's1_document', 's1_doesn', 's1_dog', 's1_dollar', 's1_domestic', 's1_don', 's1_don have', 's1_don know', 's1_don like', 's1_don think', 's1_don want', 's1_door', 's1_doro', 's1_double', 's1_doubt', 's1_down', 's1_down the', 's1_down to', 's1_dozen']


### feature engineering

In [61]:
#add feature length sentence1 and sentence2
data_tf_idf['s1_length'] = data_tf_idf['sentence1_cleaned'].apply(lambda x: len(str(x).split()))
data_tf_idf['s2_length'] = data_tf_idf['sentence2_cleaned'].apply(lambda x: len(str(x).split()))
#data_tf_idf['length_difference'] = (data_tf_idf['s1_length'] - data_tf_idf['s2_length']).abs()

#add feature to detect negation words
negation_words = set(['not', 'no', 'never', 'neither', 'nobody', 'nothing', 'nowhere', 'none', 'nor', 'cannot', "t", 'without', 'hardly', 'scarcely', 'barely', 'seldom', 'rarely'])
#add feature who many negation words in sentence1 and sentence2
def count_negation_words(text):
    words = str(text).lower().split()
    return sum(1 for word in words if word in negation_words)

data_tf_idf['s1_negation_count'] = data_tf_idf['sentence1_cleaned'].apply(count_negation_words)
data_tf_idf['s2_negation_count'] = data_tf_idf['sentence2_cleaned'].apply(count_negation_words)

In [62]:
# lexical similarity features (Jaccard and Sørensen-Dice) on cleaned tokens

def jaccard_and_dice(row):
    s1_words = set(str(row['sentence1_cleaned']).split())
    s2_words = set(str(row['sentence2_cleaned']).split())
    if not s1_words and not s2_words:
        return 0.0, 0.0
    inter = len(s1_words.intersection(s2_words))
    dice = (2 * inter) / (len(s1_words) + len(s2_words)) if (len(s1_words) + len(s2_words)) else 0.0
    return  dice

# add features
#data_tf_idf['dice_similarity'] = data_tf_idf.apply(jaccard_and_dice, axis=1)

def shared_word_percentage(row):
    s1_words = set(str(row['sentence1_cleaned']).split())
    s2_words = set(str(row['sentence2_cleaned']).split())
    if len(s1_words) == 0 or len(s2_words) == 0:
        return 0.0
    shared_words = s1_words.intersection(s2_words)
    total_words = s1_words.union(s2_words)
    return len(shared_words) / len(total_words)

data_tf_idf['shared_word_percentage_sentence1_sentence2'] = data_tf_idf.apply(shared_word_percentage, axis=1)

In [63]:
#antonyme detection
import nltk
from nltk.corpus import wordnet

nltk.download('wordnet')

def count_antonyms(row):
    s1_words = str(row['sentence1_cleaned']).lower().split()
    s2_words = str(row['sentence2_cleaned']).lower().split()
    
    antonym_count = 0
    for word in s1_words:
        antonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                if lemma.antonyms():
                    antonyms.add(lemma.antonyms()[0].name().lower())
        
        # Check if any antonym of a word in S1 exists in S2
        if any(ant in s2_words for ant in antonyms):
            antonym_count += 1
            
    return antonym_count

data_tf_idf['antonym_count'] = data_tf_idf.apply(count_antonyms, axis=1)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hamma\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [64]:
# pd.set_option('display.max_colwidth', None)

# sort by gold_label and antonym_count and only display gol label, count of lines with antonyms
# data_tf_idf.value_counts(subset=['gold_label', 'antonym_count']).sort_index()

In [65]:
#standardize engineered features
from sklearn.preprocessing import StandardScaler

cols_to_scale = [
    's1_length', 's2_length',
    's1_negation_count', 's2_negation_count',
    'antonym_count'
]

scaler = StandardScaler()
data_tf_idf[cols_to_scale] = scaler.fit_transform(data_tf_idf[cols_to_scale])

data_tf_idf.drop(columns=['sentence1_cleaned', 'sentence2_cleaned'], inplace=True)

### train models

In [66]:
#data_tf_idf = data_tf_idf.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])

feature_created = list(data_tf_idf.columns)

os.makedirs(f"../artifacts/{EXP_NUMBER}/features", exist_ok = True)

with open(f"../artifacts/{EXP_NUMBER}/features/features.json", "w") as f:
    json.dump(feature_created, f, indent=4)

In [67]:
#split the data into train and test
from sklearn.model_selection import train_test_split

#drop columns annotator_labels gold_label
cleaned_data_train = data_tf_idf.drop(columns=['gold_label'])

X_train, X_test, y_train, y_test = train_test_split(cleaned_data_train, data_tf_idf['gold_label'], test_size=0.2, random_state=42)

**dimesion reduction for distance base algorithmes**

In [68]:
# from sklearn.decomposition import TruncatedSVD

# # Fit SVD with enough components
# svd = TruncatedSVD(n_components=min(X_train.shape[1], 5000), random_state=42)
# X_train_svd = svd.fit_transform(X_train)
# X_test_svd = svd.transform(X_test)

# # Compute cumulative explained variance
# cumulative_variance = np.cumsum(svd.explained_variance_ratio_)

# # Choose number of components to explain at least 70% variance
# n_components_70 = np.searchsorted(cumulative_variance, 0.9) + 1
# print(f"Number of components to retain 90% variance: {n_components_70}")

# # Refit SVD with optimal components
# svd_opt = TruncatedSVD(n_components=n_components_70, random_state=42)
# X_train_svd = svd_opt.fit_transform(X_train)
# X_test_svd = svd_opt.transform(X_test)


In [69]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json

models = {
    # "Naive Bayes": GaussianNB(),
    # "KNeighbor Classifier": KNeighborsClassifier(),
    # "XGBoost": xgb.XGBClassifier(),
    "Random Forest": RandomForestClassifier(n_jobs=-1),
    # "Logistic Regression": LogisticRegression(multi_class='multinomial', n_jobs=-1),
    # "SVM": SVC(kernel='linear', decision_function_shape='ovo')
}

# encode labels for XGBoost (needs integer classes)
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)

for model_name, model in models.items():
    print(f"Training {model_name}...")

    if model_name == "XGBoost":
        model.fit(X_train, y_train_enc)
        pred_enc = model.predict(X_test)
        pred = label_encoder.inverse_transform(pred_enc)
        class_names = label_encoder.classes_
    elif model_name in {'SVM', 'Logistic Regression', 'KNeighbor Classifier'}:
        model.fit(X_train_svd, y_train)
        pred = model.predict(X_test_svd)
        class_names = model.classes_
    elif model_name in {'Naive Bayes', 'Random Forest'}:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        class_names = model.classes_

    # --- per-class metrics as dict ---
    per_class_metrics = {}
    for cls in class_names:
        per_class_metrics[cls] = {
            "precision": float(precision_score(y_test, pred, average=None, labels=[cls])[0]),
            "recall": float(recall_score(y_test, pred, average=None, labels=[cls])[0]),
            "f1": float(f1_score(y_test, pred, average=None, labels=[cls])[0])
        }

    # --- overall metrics ---
    accuracy = accuracy_score(y_test, pred)
    precision_weighted = precision_score(y_test, pred, average='weighted')
    recall_weighted = recall_score(y_test, pred, average='weighted')
    f1_weighted = f1_score(y_test, pred, average='weighted')

    # --- save model ---
    os.makedirs(f"../artifacts/{EXP_NUMBER}/model", exist_ok=True)
    joblib.dump(model, f"../artifacts/{EXP_NUMBER}/model/{model_name.replace(' ', '_')}_tfidf_model.pkl")
    
    os.makedirs(f"../artifacts/{EXP_NUMBER}/vectorizer", exist_ok=True)
    joblib.dump(tfidf, f"../artifacts/{EXP_NUMBER}/vectorizer/tfidf_vectorizer.pkl")

    # save metrics json
    metrics = {
        "per_class_metrics": per_class_metrics,
        "accuracy": accuracy,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }
    
    os.makedirs(f"../artifacts/{EXP_NUMBER}/metrics", exist_ok=True)
    
    #confusion matrix
    cm = confusion_matrix(y_test, pred, labels=class_names)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix for {model_name}')
    plt.savefig(f"../artifacts/{EXP_NUMBER}/metrics/{model_name.replace(' ', '_')}_confusion_matrix.png")
    plt.close()
    

    with open(f"../artifacts/{EXP_NUMBER}/metrics/{model_name.replace(' ', '_')}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)

Training Random Forest...
